In [ ]:
import numpy as np
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.tri import Triangulation

from copy import copy

In [ ]:
lvls = [0,0.1,1,2,5,10,15,20,30,50,75,100,125,150,200]
norm = colors.BoundaryNorm(lvls, 256)

In [ ]:
grid = xr.open_dataset("./grids/icon_grid_0026_R03B07_G.nc")
tri = Triangulation(np.rad2deg(grid["clon"]), np.rad2deg(grid["clat"]))
icon_area = grid["cell_area"]

In [ ]:
#ds_ctrl = xr.open_dataset("./data/" + "CTRL/" + "tot_prec_det.nc")["TOT_PREC"]
ds_satu = xr.open_dataset("./data/" + "SATU/" + "tot_prec_det.nc")["TOT_PREC"]
#ds_wilt = xr.open_dataset("./data/" + "WILT/" + "tot_prec_det.nc")["TOT_PREC"]

In [ ]:
# Define "study area"
x0, y0 = 3, 48
wx, wy = 7, 5.5

cells = ((np.rad2deg(grid["clon"]) >= x0) & (np.rad2deg(grid["clon"]) <= (x0+wx)) & 
         (np.rad2deg(grid["clat"]) >= y0) & (np.rad2deg(grid["clat"]) <= (y0+wy))).values

In [ ]:
ini_dates = [2021071012, 2021071100, 2021071212]

In [ ]:
for datetime in ini_dates:
    ds = xr.open_dataset(f"./data/CTRL/{datetime}/tot_prec_det.nc")["TOT_PREC"]

    fig, ax = plt.subplots(1, 1, figsize=(10,4), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout()

    ax.set_extent([-5, 25, 37, 62], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')

    im = ax.tricontourf(tri, ds.sel(time="2021-07-15T00") - ds.sel(time="2021-07-13T00"), levels=lvls, norm=norm)

    # Create a Rectangle patch
    rectangle = Rectangle((x0, y0), wx, wy, edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    plt.title(f"DET CTRL - Init: {datetime}")
    plt.colorbar(im, label="48h total Precipitation in kg / m^2", ticks=lvls)
    plt.savefig(f'./figs/ctrl_det_{datetime}_48h.png', dpi=300, bbox_inches='tight', format='png')
    plt.show()

In [ ]:
for datetime in ini_dates:
    ds = xr.open_dataset(f"./data/SATU/{datetime}/tot_prec_det.nc")["tot_prec"]

    fig, ax = plt.subplots(1, 1, figsize=(10,4), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout()

    ax.set_extent([-5, 25, 37, 62], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')

    im = ax.tricontourf(tri, ds.sel(time="2021-07-15T00") - ds.sel(time="2021-07-13T00"), levels=lvls, norm=norm)

    # Create a Rectangle patch
    rectangle = Rectangle((x0, y0), wx, wy, edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    plt.title(f"DET SATU - Init: {datetime}")
    plt.colorbar(im, label="48h total Precipitation in kg / m^2", ticks=lvls)
    plt.savefig(f'./figs/satu_det_{datetime}_48h.png', dpi=300, bbox_inches='tight', format='png')
    plt.show()

In [ ]:
timesteps_ctrl = {}
area_precs_ctrl = {}

for datetime in ini_dates:
    ds = xr.open_dataset(f"./data/CTRL/{datetime}/tot_prec_det.nc")["TOT_PREC"]
    area_prec = np.full(len(ds), np.nan)
    area_prec[0] = 0.

    for i in range(1,len(area_prec)):
        area_prec[i] = ((ds.isel(time=i) - ds.isel(time=i-1)) * icon_area.data).isel(ncells=cells).sum().item()
    
    timesteps_ctrl[datetime] = ds["time"]
    area_precs_ctrl[datetime] = area_prec

In [ ]:
timesteps_satu = {}
area_precs_satu = {}

for datetime in ini_dates:
    ds = xr.open_dataset(f"./data/SATU/{datetime}/tot_prec_det.nc")["tot_prec"]
    area_prec = np.full(len(ds), np.nan)
    area_prec[0] = 0.

    for i in range(1,len(area_prec)):
        area_prec[i] = ((ds.isel(time=i) - ds.isel(time=i-1)) * icon_area.data).isel(ncells=cells).sum().item()
    
    timesteps_satu[datetime] = ds["time"]
    area_precs_satu[datetime] = area_prec

In [ ]:
for datetime in ini_dates:
    line = plt.plot(timesteps_ctrl[datetime], area_precs_ctrl[datetime], label=datetime)
    plt.plot(timesteps_satu[datetime], area_precs_satu[datetime], ls="dashed", color=line[0].get_color())

plt.suptitle("Total Precipitation in Study Area")
plt.xticks(rotation=45)
plt.ylabel("Total Area Precipitation in mm/h")
plt.legend()
plt.savefig('./figs/ts_tot_prec.png', dpi=300, bbox_inches='tight', format='png')
plt.show()